# 04 — Fault Injection Attacks

## What This Is
Fault injection attacks deliberately perturb hardware to cause incorrect computation, bypassing security checks or extracting secrets. Voltage glitching, clock glitching, and laser fault injection can cause a processor to skip instructions, corrupt registers, or flip bits.

## Why It Matters
Differential Fault Analysis (DFA) on AES can recover a 128-bit key from as few as 2-3 faulty encryptions. Fault injection bypasses secure boot checks, PIN verification, and cryptographic signature validation — the foundation of smartcard and IoT device security.

## Where the Community Stands
ChipWhisperer provides voltage and clock glitching hardware. Colin O'Flynn's work at NewAE Technology has democratised this field. EMFI (Electromagnetic Fault Injection) probes from Riscure are used in commercial security labs.

## Key Attack Types
- **Instruction skip**: glitch causes CPU to skip a branch (bypass PIN check)
- **Data corruption**: flip bit in key register (DFA)
- **Boot bypass**: skip signature verification during secure boot

In [ ]:
import random
import math
from dataclasses import dataclass
from typing import List, Optional, Tuple

# ---- Simplified AES Last-Round DFA Simulation ----
# Real DFA requires specific fault models; this demonstrates the concept

SBOX = [
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16,
]
SBOX_INV = [0] * 256
for i, v in enumerate(SBOX):
    SBOX_INV[v] = i

def xtime(a: int) -> int:
    return ((a << 1) ^ 0x1B) & 0xFF if a & 0x80 else (a << 1) & 0xFF

# Toy AES: single SubBytes+AddRoundKey last round on one state byte
def aes_last_round_byte(state_byte: int, key_byte: int) -> int:
    return SBOX[state_byte] ^ key_byte

print('AES fault injection simulation setup')
print(f'SBOX[0x00] = 0x{SBOX[0]:02X}  (sanity check: should be 0x63)')

In [ ]:
# DFA attack concept: inject fault into state byte before SubBytes
# Compare faulty ciphertext with correct ciphertext to recover key

TRUE_KEY = 0xAB
STATE    = 0x42  # some intermediate state value

# Correct ciphertext
correct_ct = aes_last_round_byte(STATE, TRUE_KEY)

def dfa_attack_single_byte(correct_ct: int, state_before_fault: int,
                            fault_delta: int) -> List[int]:
    """DFA: given correct + faulty ciphertext, find key candidates."""
    faulty_state = state_before_fault ^ fault_delta
    faulty_ct    = aes_last_round_byte(faulty_state, TRUE_KEY)  # attacker observes this

    candidates = []
    for k_guess in range(256):
        # If this key is correct:
        # SBOX_INV[correct_ct ^ k_guess] should differ from SBOX_INV[faulty_ct ^ k_guess]
        # by exactly fault_delta in the state domain
        s_correct = SBOX_INV[correct_ct ^ k_guess]
        s_faulty  = SBOX_INV[faulty_ct  ^ k_guess]
        if s_correct ^ s_faulty == fault_delta:
            candidates.append(k_guess)
    return candidates, faulty_ct

fault_delta = 0x01  # single-bit fault
candidates, faulty_ct = dfa_attack_single_byte(correct_ct, STATE, fault_delta)

print(f'DFA Attack:')
print(f'  Correct ciphertext: 0x{correct_ct:02X}')
print(f'  Faulty  ciphertext: 0x{faulty_ct:02X}')
print(f'  Key candidates:     {[hex(c) for c in candidates]}')
print(f'  True key:           0x{TRUE_KEY:02X}')
print(f'  Key recovered:      {TRUE_KEY in candidates}')

## Voltage Glitching Simulation
Voltage glitching drops VCC briefly. If the glitch timing is correct, the CPU may skip one instruction. Security-critical code (PIN compare, signature verify, hash compare) is the primary target.

In [ ]:
@dataclass
class GlitchResult:
    offset_ns:   int
    width_ns:    int
    outcome:     str   # normal/glitched/crash
    instruction_skipped: Optional[int] = None

def simulate_glitch_campaign(target_fn_offset: int,
                             offset_range: Tuple[int,int],
                             width_range:  Tuple[int,int],
                             n_attempts: int,
                             rng: random.Random) -> List[GlitchResult]:
    """Simulate a voltage glitch parameter sweep.
    target_fn_offset: clock cycle of the security check instruction.
    """
    results = []
    for _ in range(n_attempts):
        offset = rng.randint(*offset_range)
        width  = rng.randint(*width_range)

        # Glitch landing window: ±5ns around target causes skip
        distance = abs(offset - target_fn_offset)
        if distance <= 5 and 2 <= width <= 15:
            outcome = 'glitched'
            skipped = target_fn_offset
        elif width > 25 or distance <= 2:
            outcome = 'crash'
            skipped = None
        else:
            outcome = 'normal'
            skipped = None

        results.append(GlitchResult(offset, width, outcome, skipped))
    return results

rng = random.Random(42)
TARGET_OFFSET = 500  # clock cycle of PIN compare instruction

results = simulate_glitch_campaign(
    target_fn_offset=TARGET_OFFSET,
    offset_range=(480, 520),
    width_range=(1, 30),
    n_attempts=500,
    rng=rng
)

from collections import Counter
outcome_counts = Counter(r.outcome for r in results)
glitched = [r for r in results if r.outcome == 'glitched']

print('Voltage Glitch Campaign Results:')
for outcome, cnt in outcome_counts.items():
    print(f'  {outcome:<10}: {cnt} ({cnt/len(results)*100:.1f}%)')

if glitched:
    print(f'\nSuccessful glitches ({len(glitched)}):')
    for g in glitched[:3]:
        print(f'  offset={g.offset_ns}ns  width={g.width_ns}ns  skipped_instr@{g.instruction_skipped}')

## Countermeasures
Defences include: voltage monitors (reset on out-of-range VCC), duplicate-and-compare (run computation twice, compare results), instruction flow integrity (check PC register against expected values), and redundant checks.

In [ ]:
# Duplicate-and-compare countermeasure
def secure_pin_verify_with_redundancy(entered_pin: str, stored_hash: str) -> bool:
    import hashlib
    """Verify PIN with redundant check — fault must hit both to succeed."""
    result_1 = hashlib.sha256(entered_pin.encode()).hexdigest() == stored_hash
    result_2 = hashlib.sha256(entered_pin.encode()).hexdigest() == stored_hash
    # A single instruction skip can only affect one check
    if result_1 != result_2:
        raise RuntimeError('Fault detected: inconsistent results!')
    return result_1 and result_2

import hashlib
CORRECT_PIN  = '1234'
STORED_HASH  = hashlib.sha256(CORRECT_PIN.encode()).hexdigest()

print('Redundant PIN verification:')
print(f'  Correct PIN:   {secure_pin_verify_with_redundancy(CORRECT_PIN, STORED_HASH)}')
print(f'  Wrong PIN:     {secure_pin_verify_with_redundancy("9999", STORED_HASH)}')

# Simulate fault detection
print('\nFault detection summary:')
print('  [x] Voltage monitor: reset on VCC < 2.7V or > 3.6V')
print('  [x] Duplicate computation: mismatch triggers tamper response')
print('  [x] Instruction flow integrity: PC checked against whitelist')
print('  [x] Constant-time comparison: timing oracle eliminated')